# 🐍 Day 34 · Python Basics — Measuring Quality: Metrics & Comparisons (RAG Evaluation)

> **Goal of this notebook:** Evaluation = compare what the system did with a **golden** answer, turn that into numbers, and check that new versions are not worse.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. Golden datasets (lists of dictionaries)
2. Hit rate and recall@k
3. Averages with `statistics`
4. Pass / fail thresholds
5. Comparing a baseline with a candidate (regression check)
6. Printing a neat results table

## 1. A golden dataset
Each golden example has a question, the **correct** source pages and what the retriever actually returned. (See `goldens/` in this folder.)

In [ ]:
goldens = [
    {"question": "JPM net income 2023?",   "expected": {"p12"},        "retrieved": ["p12", "p40", "p3"]},
    {"question": "Total revenue?",         "expected": {"p10", "p11"}, "retrieved": ["p11", "p55", "p10"]},
    {"question": "CET1 ratio?",            "expected": {"p88"},        "retrieved": ["p2", "p5", "p7"]},
    {"question": "Number of employees?",   "expected": {"p4"},         "retrieved": ["p9", "p4", "p1"]},
]
print(len(goldens), "golden examples")

## 2. Hit rate and recall@k
- **Hit@k**: did *at least one* correct page appear in the top k? (yes = 1, no = 0)
- **Recall@k**: what *fraction* of the correct pages appeared in the top k?

In [ ]:
def hit_at_k(example, k=3):
    return int(bool(example["expected"] & set(example["retrieved"][:k])))

def recall_at_k(example, k=3):
    found = example["expected"] & set(example["retrieved"][:k])
    return len(found) / len(example["expected"])

for g in goldens:
    print(f"{g['question']:<24} hit@1={hit_at_k(g, 1)}  hit@3={hit_at_k(g)}  recall@3={recall_at_k(g):.2f}")

## 3. Averages with `statistics`

In [ ]:
from statistics import mean

scores = {
    "hit@1": mean(hit_at_k(g, 1) for g in goldens),
    "hit@3": mean(hit_at_k(g) for g in goldens),
    "recall@3": mean(recall_at_k(g) for g in goldens),
}
print(scores)

## 4. Pass / fail thresholds
DeepEval marks each metric as passed when the score is at least the threshold.

In [ ]:
thresholds = {"hit@1": 0.5, "hit@3": 0.7, "recall@3": 0.7}

for metric, value in scores.items():
    passed = value >= thresholds[metric]
    print(f"{metric:<9} {value:.2f}  (needs {thresholds[metric]:.2f})  {'✅ PASS' if passed else '❌ FAIL'}")

## 5. 🔗 Regression check: baseline vs candidate
Before releasing a new version (for example adding a re-ranker), compare it with the old one. `evals/compare.py` does this with `baselines/*.json`.

In [ ]:
baseline  = {"hit@3": 0.75, "recall@3": 0.70, "faithfulness": 0.88, "latency_s": 2.1}
candidate = {"hit@3": 0.82, "recall@3": 0.78, "faithfulness": 0.84, "latency_s": 2.6}
lower_is_better = {"latency_s"}
tolerance = 0.02                                    # small changes are just noise

print(f"{'metric':<13}{'baseline':>9}{'candidate':>11}{'change':>9}  verdict")
for m in baseline:
    change = candidate[m] - baseline[m]
    worse = change > tolerance if m in lower_is_better else change < -tolerance
    verdict = "🔻 REGRESSION" if worse else "ok"
    print(f"{m:<13}{baseline[m]:>9.2f}{candidate[m]:>11.2f}{change:>+9.2f}  {verdict}")

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Compute **precision@3** = correct pages in top 3 ÷ 3 for every golden example.
2. Raise the `hit@1` threshold to 0.8 — does it still pass?
3. Add `"cost_usd"` (lower is better) to the baseline and candidate and re-run the comparison.

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
def precision_at_k(example, k=3):
    return len(example["expected"] & set(example["retrieved"][:k])) / k
print([round(precision_at_k(g), 2) for g in goldens])

thresholds["hit@1"] = 0.8   # then re-run section 4 -> FAIL

baseline["cost_usd"], candidate["cost_usd"] = 0.010, 0.014
lower_is_better.add("cost_usd")
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Golden example | `{'question': q, 'expected': {...}, 'retrieved': [...]}` |
| Hit@k | `int(bool(expected & set(retrieved[:k])))` |
| Recall@k | `len(found) / len(expected)` |
| Average | `statistics.mean(values)` |
| Threshold | `passed = value >= threshold` |
| Signed number | `f'{change:+.2f}'` |

## ➡️ What's next?
You now know the Python used in **Day_34_RAG_Evaluation_DeepEval**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀